In [ ]:
from pathlib import Path
import torch
from gelos.gelosdatamodule import GELOSDataModule
import yaml
from lightning.pytorch import Trainer
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
from terratorch.tasks import EmbeddingGenerationTask
from gelos.embedding_generation import LenientEmbeddingGenerationTask
from lightning.pytorch.cli import instantiate_class
import numpy as np
from matplotlib import transforms, patches
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
from matplotlib import transforms, patches
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import yaml
from pathlib import Path
from gelos.generation import setup_embedding_run

In [ ]:
yaml_path = Path("/app/configs/exp001_prithvi300.yaml")
raw_data_dir = Path("/app/data/raw")
embedding_dir = Path("/app/data/interim")
with open(yaml_path, "r") as f:
    yaml_config = yaml.safe_load(f)
version = yaml_config['data_version']
figures_dir = Path("/app/reports/figures")

## Set up the embedding run

In [ ]:
datamodule, task, trainer, output_dir = setup_embedding_run(
    yaml_path=yaml_path,
    raw_data_dir=raw_data_dir,
    embedding_dir=embedding_dir,
)

## Run embedding generation step by step

### Inspect outputs of dataset

In [ ]:
datamodule.setup(stage="predict")

In [ ]:
print(f"Dataset sensors and bands: {datamodule.dataset.bands}")

In [ ]:
for k, v in datamodule.dataset[0].items():
    if k == "image" and isinstance(v, dict):
        for sensor, data in v.items():
            print(sensor, data.shape)
    elif k == "image":
        print (k, v.shape)
    else:
        print(k, v)

In [ ]:
plot = datamodule.dataset.plot(datamodule.dataset[0])

## Generate Embeddings

### Generate one embedding and inspect

In [ ]:
sample = datamodule.dataset[0] # get one sample from the dataset
image = sample["image"].unsqueeze(0) # add dummy batch dimension
final_layer_embedding = task.model(image)[-1]

#### Visualize embeddings and slicing

This section is not necessary for the embedding pipeline, but is a useful check on what patches are being sliced using the slicing args.

Currently set up for Prithvi EO 300M embedding shapes.

In [ ]:
# Customize visualization paramters
vis_embed_depth = 128
n_per_grid = 36
date_ranges = ["Jan-Mar", "Apr-Jun", "Jul-Sep", "Oct-Dec"]

In [ ]:
patch_tokens_flat = final_layer_embedding.reshape(-1, final_layer_embedding.shape[-1])
vis_embed_depth = min(vis_embed_depth, patch_tokens_flat.shape[-1])
patch_tokens_sample = patch_tokens_flat[:, 0:vis_embed_depth].detach().cpu().numpy()
first_feature = patch_tokens_flat[:, 0].detach().cpu().numpy()
start_index = 1 if has_cls else 0
groups_3d = [patch_tokens_sample[i : i + n_per_grid] for i in range(start_index, patch_tokens_sample.shape[0], n_per_grid)]

neon_colors = ['#FF00FF', '#39FF14', '#FF0033']

# Create a custom colormap from the list of colors
# N specifies the number of color quantization levels for a smoother gradient
custom_neon_cmap = ListedColormap(neon_colors, name='custom_neon')

grid_side = int(np.ceil(np.sqrt(n_per_grid)))
cells_per_grid = grid_side * grid_side
start_index = 1 if has_cls else 0
groups = [first_feature[i : i + n_per_grid] for i in range(start_index, first_feature.shape[0], n_per_grid)]
group_count = len(groups)
cols = group_count + 1
rows = 1
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
axes_flat = axes.ravel() if isinstance(axes, np.ndarray) else np.array([axes])
mesh = None
x_skew = 0
y_skew = -25
skew_base = transforms.Affine2D().skew_deg(x_skew, y_skew)

highlight_dict = {}
cls_first_patch_highlights = []
# only works with prithvi configs, since this is capable of showing all extraction strategies including CLS
for extraction_strategy, extraction_args in yaml_config['embedding_extraction_strategies'].items():
    slice_args = extraction_args['slice_args']
    title = extraction_args['title']
    highlight_spec = slice_args[0].copy()
    if highlight_spec["stop"] is None:
        highlight_spec["stop"] = n_per_grid * group_count
    if has_cls:
        highlight_spec["start"] -= start_index 
        highlight_spec["stop"] -= start_index
    if highlight_spec["stop"] - highlight_spec["start"] == n_per_grid and highlight_spec["step"] == 1:
        highlight_spec["outline_plot"] = int(round((highlight_spec["stop"] - highlight_spec["start"]) / n_per_grid))
    highlight_dict[title] = highlight_spec
    if highlight_spec["start"] < 0:
        cls_first_patch_highlights.append(title)
    print(title, highlight_spec)
# cmap = plt.get_cmap("cool", max(len(highlight_dict), 1))
cmap = ListedColormap(neon_colors, name='custom_neon')
for idx, (name, spec) in enumerate(highlight_dict.items()):
    spec["color"] = cmap(idx)

outline_map = {}
highlight_map = {}
for name, spec in highlight_dict.items():
    outline_idx = spec.get("outline_plot")
    if outline_idx is not None and 0 <= outline_idx < group_count:
        outline_map.setdefault(outline_idx, []).append(spec["color"])
        continue
    if spec.get("start", 0) < 0:
        continue
    abs_start = max(0, spec.get("start", 0))
    abs_stop = min(spec.get("stop", first_feature.shape[0]), first_feature.shape[0])
    abs_step = max(1, spec.get("step", 1))
    for token_idx in range(abs_start, abs_stop, abs_step):
        group_idx = token_idx // n_per_grid
        if group_idx >= group_count:
            break
        highlight_map.setdefault(group_idx, []).append((token_idx % n_per_grid, spec["color"], name))

legend_patches = [patches.Patch(color=spec["color"], label=name) for name, spec in highlight_dict.items()]

# first patch (single square) on the left, sized like a normal patch cell and centered
first_patch_value = first_feature[start_index]
single_patch_grid = np.full((grid_side, grid_side), np.nan)
center_idx = grid_side // 2
single_patch_grid[center_idx, center_idx] = first_patch_value
ax_first = axes_flat[0]
mesh = ax_first.pcolormesh(single_patch_grid, cmap='coolwarm', edgecolors='none', linewidth=2, shading='auto',
                         transform=skew_base + ax_first.transData, antialiased=True)
mesh.set_clip_on(False)
for cls_name in cls_first_patch_highlights:
    color = highlight_dict[cls_name]["color"]
    rect = patches.Rectangle((center_idx, center_idx), 1, 1, fill=False, edgecolor=color, linewidth=2,
                             transform=skew_base + ax_first.transData, clip_on=False)
    ax_first.add_patch(rect)
ax_first.set_title('CLS Token (Prithvi)', y=.7)
ax_first.set_xticks([])
ax_first.set_yticks([])
ax_first.set_aspect('equal')
for spine in ax_first.spines.values():
    spine.set_visible(False)

for idx, group_values in enumerate(groups):
    chunk = group_values.copy()
    chunk.shape
    if chunk.shape[0] < n_per_grid:
        chunk = np.concatenate([chunk, np.full(n_per_grid - chunk.shape[0], np.nan, dtype=chunk.dtype)])
    if chunk.shape[0] < cells_per_grid:
        chunk = np.concatenate([chunk, np.full(cells_per_grid - chunk.shape[0], np.nan, dtype=chunk.dtype)])
    grid = chunk.reshape(grid_side, grid_side)
    ax = axes_flat[idx + 1]
    mesh = ax.pcolormesh(grid, cmap='coolwarm', edgecolors='white', linewidth=2, shading='auto',
                         transform=skew_base + ax.transData, antialiased=True)
    mesh.set_clip_on(False)
    for cell_idx, color, _name in highlight_map.get(idx, []):
        row = cell_idx // grid_side
        col = cell_idx % grid_side
        rect = patches.Rectangle((col, row), 1, 1, fill=False, edgecolor=color, linewidth=2,
                                 transform=skew_base + ax.transData, clip_on=False)
        ax.add_patch(rect)
    for outline_color in outline_map.get(idx, []):
        outline = patches.Rectangle((0, 0), grid_side, grid_side, fill=False, edgecolor=outline_color, linewidth=2,
                                    transform=skew_base + ax.transData, clip_on=False)
        ax.add_patch(outline)
    ax.set_title(f'{date_ranges[idx]}', y=0.95)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes_flat[group_count + 1:]:
    ax.axis('off')

fig.subplots_adjust(wspace=-0.60, hspace=-0.60)
if legend_patches:
    fig.legend(
        handles=legend_patches,
        loc="center right",
        bbox_to_anchor=(1, 0.5),
        # ncol=max(len(legend_patches), 1),
        title="Embedding Extraction Strategies"
    )
plt.suptitle("Example Embedding Outputs and Extraction Strategies")
fig.canvas.draw()
plt.savefig(figures_dir / version / "example_embedding_outputs.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()


### Generate all embeddings

In [ ]:
# Generate Embeddings
trainer.predict(model=task, datamodule=datamodule)